In [1]:
import numpy as np
import pandas as pd
# Necessary imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta
import shutil

df = pd.read_csv("prob_output_stocks_2024.csv")# output from model inference
df["Date"] = pd.to_datetime(df["Date"])
df.sort_values(by=["Date"])


,Unnamed: 0,Date,Prob,Label,Symbol
35362,35362,2023-11-29,0.462904,1,IVZ
32708,32708,2023-11-29,0.555739,-1,PG
27261,27261,2023-11-29,0.543904,-1,APTV
48009,48009,2023-11-29,0.583089,1,SYY
15054,15054,2023-11-29,0.671041,1,AEE
...,...,...,...,...,...
59311,59311,2024-05-23,0.699339,1,XEL
7749,7749,2024-05-23,0.759419,-1,PWR
41146,41146,2024-05-23,0.445456,1,EXR
29562,29562,2024-05-23,0.254179,-1,BX


,Unnamed: 0,Date,Prob,Label,Symbol
2366,2366,2024-03-28,0.484816,1,EMN
10932,10932,2024-03-28,0.484728,-1,AXP
10941,10941,2024-03-28,0.392360,-1,IRM
18971,18971,2024-03-28,0.463787,1,MET
663,663,2024-03-28,0.531242,-1,SRE
...,...,...,...,...,...
5424,5424,2024-05-23,0.556375,-1,CRL
5462,5462,2024-05-23,0.555854,1,EXPD
5509,5509,2024-05-23,0.565640,-1,CME
12322,12322,2024-05-23,0.553600,-1,HSY


In [2]:
# Add in additional features here if needed
def data_generation(data_path, freq):
    if freq == 'D':
        data = pd.read_csv(data_path, index_col=0, parse_dates=True)
    elif freq == 'H':
        data = pd.read_csv(data_path, index_col=0)
        incr = 1e-9
        unix_start = datetime(1970, 1, 1)
        date = []
        for i in range(len(data)):
            date.append(unix_start + timedelta(seconds=incr*data.index[i]))
        data.index = date
        for col in ['open', 'high', 'low', 'close']:
            data[col] = data[col].astype(float) * incr
    # Add features here
    
    data.columns = data.columns.str.lower()
    return data

In [3]:
# Get the list of S&P 500 companies from Wikipedia
sp500_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
table = pd.read_html(sp500_url)
sp500_df = table[0]
tickers = sp500_df['Symbol'].tolist()
len(tickers)

503

In [4]:
n = 198
exc_tickers = []
inc_tickers = []
error_count = 0
error_list = []
tdf = pd.DataFrame([])
for i, ticker in enumerate(tickers):
    print(f"Working at {ticker, i}")
    try:
        d = data_generation(f'data-SPYstock/{ticker}.csv', freq='D')
        pd_dt = pd.to_datetime(datetime(2023,9,1))
        d = d[(d.index>pd_dt) & (d.index<=datetime(2024,6,17))]
        if len(d)==n:
            inc_tickers.append(ticker)
            d["Symbol"] = ticker
            d["close_d1"] = d["close"].shift(-1)
            d["close_d1_p5"] = d["close_d1"].shift(-5)
            d["true_label"] = np.where(d["close_d1_p5"]>d["close_d1"], 1, -1)
            d.dropna(inplace=True)
            tdf = pd.concat([tdf, d])
        else:
            exc_tickers.append(ticker)
            print(f"Data length mismatch for {ticker}")
    except Exception as e:
        print(f"An error occurred: {e}")
        print(f"Error generating {ticker}")
        error_count += 1
        error_list.append(ticker)

Working at ('MMM', 0)
Working at ('AOS', 1)
Working at ('ABT', 2)
Working at ('ABBV', 3)
Working at ('ACN', 4)
Working at ('ADBE', 5)
Working at ('AMD', 6)
Working at ('AES', 7)
Working at ('AFL', 8)
Working at ('A', 9)
Working at ('APD', 10)
Working at ('ABNB', 11)
Working at ('AKAM', 12)
Working at ('ALB', 13)
Working at ('ARE', 14)
Working at ('ALGN', 15)
Working at ('ALLE', 16)
Working at ('LNT', 17)
Working at ('ALL', 18)
Working at ('GOOGL', 19)
Working at ('GOOG', 20)
Working at ('MO', 21)
Working at ('AMZN', 22)
Working at ('AMCR', 23)
Working at ('AEE', 24)
Working at ('AAL', 25)
Working at ('AEP', 26)
Working at ('AXP', 27)
Working at ('AIG', 28)
Working at ('AMT', 29)
Working at ('AWK', 30)
Working at ('AMP', 31)
Working at ('AME', 32)
Working at ('AMGN', 33)
Working at ('APH', 34)
Working at ('ADI', 35)
Working at ('ANSS', 36)
Working at ('AON', 37)
Working at ('APA', 38)
Working at ('AAPL', 39)
Working at ('AMAT', 40)
Working at ('APTV', 41)
Working at ('ACGL', 42)
Working

In [5]:
tdf.to_csv("./true_labels_stocks_2024.csv")

#tdf

In [6]:
tdf

,open,high,low,close,adj close,volume,Symbol,close_d1,close_d1_p5,true_label
Date,,,,,,,,,,
2023-09-05,89.297661,90.518394,88.946487,89.339462,85.921562,5817583,MMM,88.938126,84.489967,-1
2023-09-06,89.063545,89.306023,88.076920,88.938126,85.535576,3210423,MMM,88.586960,85.551842,-1
2023-09-07,88.779266,89.289299,87.759193,88.586960,85.197853,3476652,MMM,88.829430,84.498329,-1
2023-09-08,88.695648,88.954849,88.143814,88.829430,85.431046,3317824,MMM,90.209030,84.481606,-1
2023-09-11,89.464882,90.484947,89.289299,90.209030,86.757858,3398314,MMM,89.598663,83.804352,-1
...,...,...,...,...,...,...,...,...,...,...
2024-06-03,170.039993,174.270004,169.979996,172.369995,172.369995,2142700,ZTS,171.880005,178.539993,1
2024-06-04,170.929993,172.259995,169.660004,171.880005,171.880005,1304400,ZTS,175.820007,175.850006,1
2024-06-05,172.899994,176.729996,172.899994,175.820007,175.820007,1870800,ZTS,176.779999,171.320007,-1


In [8]:
tdf["Symbol"].value_counts()

Symbol
MMM     192
MS      192
NDSN    192
NI      192
NKE     192
       ... 
EBAY    192
ETN     192
EMN     192
DD      192
ZTS     192
Name: count, Length: 497, dtype: int64

In [12]:
#historic data

n = 250
exc_tickers = []
inc_tickers = []
error_count = 0
error_list = []
tdf = pd.DataFrame([])
for i, ticker in enumerate(tickers):
    print(f"Working at {ticker, i}")
    try:
        d = data_generation(f'data-SPYstock/{ticker}.csv', freq='D')
        pd_dt = pd.to_datetime(datetime(2023,1,1))
        d = d[(d.index>pd_dt) & (d.index<=datetime(2023,12,31))]
        # print(len(d))
        if len(d)==n:
            inc_tickers.append(ticker)
            d["Symbol"] = ticker
            d["close_d1"] = d["close"].shift(-1)
            d["close_d1_p5"] = d["close_d1"].shift(-5)
            d["return_d5"] = (d["close_d1_p5"] - d["close_d1"] )/d["close_d1"] 
            d.dropna(inplace=True)
            tdf = pd.concat([tdf, d])
        else:
            exc_tickers.append(ticker)
            print(f"Data length mismatch for {ticker}")
    except Exception as e:
        print(f"An error occurred: {e}")
        print(f"Error generating {ticker}")
        error_count += 1
        error_list.append(ticker)

Working at ('MMM', 0)
Working at ('AOS', 1)
Working at ('ABT', 2)
Working at ('ABBV', 3)
Working at ('ACN', 4)
Working at ('ADBE', 5)
Working at ('AMD', 6)
Working at ('AES', 7)
Working at ('AFL', 8)
Working at ('A', 9)
Working at ('APD', 10)
Working at ('ABNB', 11)
Working at ('AKAM', 12)
Working at ('ALB', 13)
Working at ('ARE', 14)
Working at ('ALGN', 15)
Working at ('ALLE', 16)
Working at ('LNT', 17)
Working at ('ALL', 18)
Working at ('GOOGL', 19)
Working at ('GOOG', 20)
Working at ('MO', 21)
Working at ('AMZN', 22)
Working at ('AMCR', 23)
Working at ('AEE', 24)
Working at ('AAL', 25)
Working at ('AEP', 26)
Working at ('AXP', 27)
Working at ('AIG', 28)
Working at ('AMT', 29)
Working at ('AWK', 30)
Working at ('AMP', 31)
Working at ('AME', 32)
Working at ('AMGN', 33)
Working at ('APH', 34)
Working at ('ADI', 35)
Working at ('ANSS', 36)
Working at ('AON', 37)
Working at ('APA', 38)
Working at ('AAPL', 39)
Working at ('AMAT', 40)
Working at ('APTV', 41)
Working at ('ACGL', 42)
Working

In [13]:
tdf.to_csv("./historic_data_stocks.csv")

tdf

,open,high,low,close,adj close,volume,Symbol,close_d1,close_d1_p5,return_d5
Date,,,,,,,,,,
2023-01-03,101.605354,102.541809,100.643814,102.399666,94.329193,3124909,MMM,104.640465,107.959869,0.031722
2023-01-04,103.135452,104.757523,102.600334,104.640465,96.393394,3312561,MMM,102.809364,108.152176,0.051968
2023-01-05,103.854515,104.155518,102.391304,102.809364,94.706612,3117494,MMM,105.953178,108.285950,0.022017
2023-01-06,104.230766,106.295990,103.469902,105.953178,97.602646,2890732,MMM,106.011703,105.852844,-0.001499
2023-01-09,106.187294,108.244148,105.443146,106.011703,97.656563,3434075,MMM,107.132111,102.633781,-0.041989
...,...,...,...,...,...,...,...,...,...,...
2023-12-14,198.000000,201.919998,198.000000,200.089996,199.069595,3044400,ZTS,196.289993,194.979996,-0.006674
2023-12-15,199.410004,199.410004,193.970001,196.289993,195.288971,4058300,ZTS,196.720001,195.500000,-0.006202
2023-12-18,197.809998,198.139999,195.600006,196.720001,195.716797,1543100,ZTS,198.080002,196.899994,-0.005957
